In [2]:
#!/usr/bin/env python3
"""
scrape_hn_claude.py

Collect public Hacker News conversation about Anthropic's Claude over the past 6 months.

Outputs:
  - data/hn_claude_stories.csv
  - data/hn_claude_comments.csv
  - data/hn_claude_summary.json

Dependencies:
  pip install requests pandas
"""

from __future__ import annotations

import html
import json
import os
import re
import time
from collections import Counter
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


ALGOLIA_BASE = "https://hn.algolia.com/api/v1/search_by_date"
HN_ITEM_BASE = "https://hacker-news.firebaseio.com/v0/item/{}.json"

OUTPUT_DIR = "data"
REQUEST_TIMEOUT = 30
PAGE_SIZE = 1000

# Broad enough to catch most Claude AI discussion on HN, then we post-filter.
SEARCH_QUERIES = [
    "Claude",
    "\"Anthropic Claude\"",
    "\"Claude 3\"",
    "\"Claude 3.5\"",
    "\"Claude 3.7\"",
]

# Terms that strongly indicate the discussion is about the AI model, not a person named Claude.
AI_CONTEXT_TERMS = [
    "anthropic", "llm", "model", "ai", "assistant", "sonnet", "opus", "haiku",
    "prompt", "mcp", "api", "context window", "artifacts", "chatbot", "reasoning"
]

# Optional false-positive terms you may want to exclude.
LIKELY_PERSON_TERMS = [
    "monet", "debussy", "shannon", "claude shannon", "claude monet"
]


def make_session() -> requests.Session:
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retries, pool_connections=20, pool_maxsize=20)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({
        "User-Agent": "hn-claude-scraper/1.0 (+public research script)"
    })
    return session


def clean_html_text(value: Optional[str]) -> str:
    if not value:
        return ""
    text = html.unescape(value)
    text = re.sub(r"<pre><code>.*?</code></pre>", " ", text, flags=re.S | re.I)
    text = re.sub(r"<code>.*?</code>", " ", text, flags=re.S | re.I)
    text = re.sub(r"<a [^>]*>(.*?)</a>", r"\1", text, flags=re.S | re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def unix_now() -> int:
    return int(datetime.now(timezone.utc).timestamp())


def six_months_ago_ts() -> int:
    # Simple and robust rolling 6-month window approximation.
    return int((datetime.now(timezone.utc) - timedelta(days=183)).timestamp())


def contains_ai_context(text: str) -> bool:
    t = text.lower()
    if "claude" not in t:
        return False

    if any(term in t for term in LIKELY_PERSON_TERMS):
        # Allow override if AI context is also present.
        if not any(term in t for term in AI_CONTEXT_TERMS):
            return False

    if "anthropic claude" in t:
        return True

    return any(term in t for term in AI_CONTEXT_TERMS)


def canonical_story_url(hit: Dict[str, Any]) -> Optional[str]:
    # Algolia may store the submitted URL in "url". If it is an Ask HN thread, use HN item URL.
    url = hit.get("url")
    if url:
        return url
    object_id = hit.get("objectID")
    if object_id:
        return f"https://news.ycombinator.com/item?id={object_id}"
    return None


def get_json(session: requests.Session, url: str, params: Optional[Dict[str, Any]] = None) -> Any:
    resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def fetch_algolia_hits(
    session: requests.Session,
    query: str,
    tag: str,
    cutoff_ts: int,
    sleep_s: float = 0.15,
) -> List[Dict[str, Any]]:
    """
    Fetch hits from Algolia search_by_date, newest first, until we pass cutoff_ts.
    """
    page = 0
    results: List[Dict[str, Any]] = []

    while True:
        params = {
            "query": query,
            "tags": tag,
            "hitsPerPage": PAGE_SIZE,
            "page": page,
            # Numeric filter ensures we only request the last 6 months.
            "numericFilters": f"created_at_i>={cutoff_ts}",
        }

        data = get_json(session, ALGOLIA_BASE, params=params)
        hits = data.get("hits", [])
        if not hits:
            break

        results.extend(hits)

        nb_pages = data.get("nbPages", 0)
        page += 1
        if page >= nb_pages:
            break

        time.sleep(sleep_s)

    return results


def fetch_hn_item(session: requests.Session, item_id: int) -> Optional[Dict[str, Any]]:
    try:
        return get_json(session, HN_ITEM_BASE.format(item_id))
    except requests.RequestException:
        return None


def dedupe_hits(hits: Iterable[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    deduped = []
    for hit in hits:
        oid = hit.get("objectID")
        if oid in seen:
            continue
        seen.add(oid)
        deduped.append(hit)
    return deduped


def story_row_from_hit_and_item(hit: Dict[str, Any], item: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    text = clean_html_text(hit.get("story_text") or hit.get("comment_text") or hit.get("title") or "")
    title = clean_html_text(hit.get("title") or hit.get("story_title") or "")
    created_at_i = hit.get("created_at_i")

    kids = item.get("kids") if isinstance(item, dict) else None
    descendants = item.get("descendants") if isinstance(item, dict) else None

    return {
        "hn_id": int(hit["objectID"]),
        "type": "story",
        "query_match_title": title,
        "author": hit.get("author"),
        "created_at": hit.get("created_at"),
        "created_at_i": created_at_i,
        "title": title,
        "story_text": text,
        "url": canonical_story_url(hit),
        "points": item.get("score") if item else hit.get("points"),
        "num_comments": item.get("descendants") if item else hit.get("num_comments"),
        "replies_kids_count": len(kids) if isinstance(kids, list) else None,
        "official_type": item.get("type") if item else None,
        "official_dead": item.get("dead") if item else None,
        "official_deleted": item.get("deleted") if item else None,
        "match_source": "algolia_search",
    }


def comment_row_from_hit_and_item(hit: Dict[str, Any], item: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    comment_text = clean_html_text(hit.get("comment_text") or "")
    story_title = clean_html_text(hit.get("story_title") or "")
    title = clean_html_text(hit.get("title") or "")
    created_at_i = hit.get("created_at_i")

    kids = item.get("kids") if isinstance(item, dict) else None

    return {
        "hn_id": int(hit["objectID"]),
        "type": "comment",
        "author": hit.get("author"),
        "created_at": hit.get("created_at"),
        "created_at_i": created_at_i,
        "comment_text": comment_text,
        "story_id": hit.get("story_id"),
        "story_title": story_title,
        "parent_id": item.get("parent") if item else hit.get("parent_id"),
        "reply_count": len(kids) if isinstance(kids, list) else 0,
        "official_dead": item.get("dead") if item else None,
        "official_deleted": item.get("deleted") if item else None,
        "match_source": "algolia_search",
        "title": title,
    }


def relevant_story(hit: Dict[str, Any]) -> bool:
    blob = " ".join([
        clean_html_text(hit.get("title") or ""),
        clean_html_text(hit.get("story_text") or ""),
        clean_html_text(hit.get("url") or ""),
    ])
    return contains_ai_context(blob)


def relevant_comment(hit: Dict[str, Any]) -> bool:
    blob = " ".join([
        clean_html_text(hit.get("comment_text") or ""),
        clean_html_text(hit.get("story_title") or ""),
        clean_html_text(hit.get("title") or ""),
    ])
    return contains_ai_context(blob)


def summarize(stories_df: pd.DataFrame, comments_df: pd.DataFrame) -> Dict[str, Any]:
    summary: Dict[str, Any] = {}

    summary["generated_at_utc"] = datetime.now(timezone.utc).isoformat()
    summary["window_days"] = 183
    summary["story_count"] = int(len(stories_df))
    summary["comment_count"] = int(len(comments_df))
    summary["unique_story_authors"] = int(stories_df["author"].nunique()) if not stories_df.empty else 0
    summary["unique_comment_authors"] = int(comments_df["author"].nunique()) if not comments_df.empty else 0

    if not stories_df.empty:
        summary["total_story_points"] = int(stories_df["points"].fillna(0).sum())
        summary["total_story_comments"] = int(stories_df["num_comments"].fillna(0).sum())
        top_stories = (
            stories_df.sort_values(["points", "num_comments"], ascending=[False, False])
            .head(20)[["hn_id", "title", "author", "points", "num_comments", "created_at", "url"]]
            .to_dict(orient="records")
        )
        summary["top_stories"] = top_stories

        stories_df["date"] = pd.to_datetime(stories_df["created_at"], utc=True).dt.date.astype(str)
        story_volume = stories_df.groupby("date").size().sort_index()
        summary["story_volume_by_day"] = story_volume.to_dict()

    if not comments_df.empty:
        comments_df["date"] = pd.to_datetime(comments_df["created_at"], utc=True).dt.date.astype(str)
        comment_volume = comments_df.groupby("date").size().sort_index()
        summary["comment_volume_by_day"] = comment_volume.to_dict()

        commenters = Counter(comments_df["author"].dropna().tolist()).most_common(20)
        summary["top_commenters"] = [{"author": author, "count": count} for author, count in commenters]

    return summary


def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    session = make_session()
    cutoff_ts = six_months_ago_ts()

    print(f"[INFO] Collecting Hacker News stories/comments about Claude since UTC ts={cutoff_ts}")

    raw_story_hits: List[Dict[str, Any]] = []
    raw_comment_hits: List[Dict[str, Any]] = []

    for query in SEARCH_QUERIES:
        print(f"[INFO] Searching stories for query={query}")
        raw_story_hits.extend(fetch_algolia_hits(session, query=query, tag="story", cutoff_ts=cutoff_ts))

        print(f"[INFO] Searching comments for query={query}")
        raw_comment_hits.extend(fetch_algolia_hits(session, query=query, tag="comment", cutoff_ts=cutoff_ts))

    raw_story_hits = dedupe_hits(raw_story_hits)
    raw_comment_hits = dedupe_hits(raw_comment_hits)

    print(f"[INFO] Raw deduped stories: {len(raw_story_hits)}")
    print(f"[INFO] Raw deduped comments: {len(raw_comment_hits)}")

    filtered_story_hits = [h for h in raw_story_hits if relevant_story(h)]
    filtered_comment_hits = [h for h in raw_comment_hits if relevant_comment(h)]

    print(f"[INFO] Relevant stories after AI-context filter: {len(filtered_story_hits)}")
    print(f"[INFO] Relevant comments after AI-context filter: {len(filtered_comment_hits)}")

    stories_rows: List[Dict[str, Any]] = []
    comments_rows: List[Dict[str, Any]] = []

    # Enrich stories with official HN item data.
    for i, hit in enumerate(filtered_story_hits, start=1):
        item_id = int(hit["objectID"])
        item = fetch_hn_item(session, item_id)
        stories_rows.append(story_row_from_hit_and_item(hit, item))
        if i % 100 == 0:
            print(f"[INFO] Enriched {i}/{len(filtered_story_hits)} stories")
        time.sleep(0.03)

    # Enrich comments with official HN item data.
    for i, hit in enumerate(filtered_comment_hits, start=1):
        item_id = int(hit["objectID"])
        item = fetch_hn_item(session, item_id)
        comments_rows.append(comment_row_from_hit_and_item(hit, item))
        if i % 250 == 0:
            print(f"[INFO] Enriched {i}/{len(filtered_comment_hits)} comments")
        time.sleep(0.03)

    stories_df = pd.DataFrame(stories_rows)
    comments_df = pd.DataFrame(comments_rows)

    if not stories_df.empty:
        stories_df = stories_df.sort_values("created_at_i", ascending=False).reset_index(drop=True)
    if not comments_df.empty:
        comments_df = comments_df.sort_values("created_at_i", ascending=False).reset_index(drop=True)

    stories_csv = os.path.join(OUTPUT_DIR, "hn_claude_stories.csv")
    comments_csv = os.path.join(OUTPUT_DIR, "hn_claude_comments.csv")
    summary_json = os.path.join(OUTPUT_DIR, "hn_claude_summary.json")

    stories_df.to_csv(stories_csv, index=False)
    comments_df.to_csv(comments_csv, index=False)

    summary = summarize(stories_df.copy(), comments_df.copy())
    with open(summary_json, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"[DONE] Wrote {stories_csv}")
    print(f"[DONE] Wrote {comments_csv}")
    print(f"[DONE] Wrote {summary_json}")
    print(json.dumps({
        "stories": len(stories_df),
        "comments": len(comments_df),
        "summary_file": summary_json
    }, indent=2))


if __name__ == "__main__":
    main()

[INFO] Collecting Hacker News stories/comments about Claude since UTC ts=1759535244
[INFO] Searching stories for query=Claude
[INFO] Searching comments for query=Claude
[INFO] Searching stories for query="Anthropic Claude"
[INFO] Searching comments for query="Anthropic Claude"
[INFO] Searching stories for query="Claude 3"
[INFO] Searching comments for query="Claude 3"
[INFO] Searching stories for query="Claude 3.5"
[INFO] Searching comments for query="Claude 3.5"
[INFO] Searching stories for query="Claude 3.7"
[INFO] Searching comments for query="Claude 3.7"
[INFO] Raw deduped stories: 1120
[INFO] Raw deduped comments: 1291
[INFO] Relevant stories after AI-context filter: 724
[INFO] Relevant comments after AI-context filter: 1019
[INFO] Enriched 100/724 stories
[INFO] Enriched 200/724 stories
[INFO] Enriched 300/724 stories
[INFO] Enriched 400/724 stories
[INFO] Enriched 500/724 stories
[INFO] Enriched 600/724 stories
[INFO] Enriched 700/724 stories
[INFO] Enriched 250/1019 comments
[I